In [1]:
import pandas as pd
import numpy as np
import holidays


! pip install holidays -q

dfTrain = pd.read_csv('training_data.csv', encoding='latin1')
dfTest = pd.read_csv('test_data.csv', encoding='latin1')



print("A criar features de Data e Hora...")

for df in [dfTrain, dfTest]:
    # Converter para datetime
    df['record_date'] = pd.to_datetime(df['record_date'])

    df['year'] = df['record_date'].dt.year
    df['month'] = df['record_date'].dt.month
    df['day'] = df['record_date'].dt.day
    df['hour'] = df['record_date'].dt.hour
    df['dayOfWeek'] = df['record_date'].dt.dayofweek


    df['isRushHour'] = ((df['hour'] >= 7) & (df['hour'] <= 10) | 
                        (df['hour'] >= 16) & (df['hour'] <= 21)).astype(int)
    
   
    df['IS_WEEKEND'] = (df['dayOfWeek'] >= 5).astype(int)

  
    anos = df['year'].unique()
    feriados_portugal = holidays.Portugal(years=anos)
    df['is_holiday'] = df['record_date'].dt.date.isin(feriados_portugal).astype(int)

    # features ciclicas, supostamente melhoram o desempenho de certos modelos mas é uma questão de teste.
    # Por exemplo, 23 horas e 1 hora estão próximas, mas numericamente são distantes (23 e 1). O modelo pode interpretar mal essa distância.
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
    df['dayOfWeek_sin'] = np.sin(2 * np.pi * df['dayOfWeek'] / 7.0)
    df['dayOfWeek_cos'] = np.cos(2 * np.pi * df['dayOfWeek'] / 7.0)
    
    bins_dia = [0, 6, 12, 18, 24]
    labels_dia = ['Madrugada', 'Manha', 'Tarde', 'Noite']
    df['Parte_do_Dia'] = pd.cut(df['hour'], bins=bins_dia, labels=labels_dia, right=False)
    
    def get_season(month):
        if month in [12, 1, 2]: return 'Inverno'
        if month in [3, 4, 5]: return 'Primavera'
        if month in [6, 7, 8]: return 'Verao'
        return 'Outono'
    df['Estacao_do_Ano'] = df['month'].apply(get_season)


print("A mapear features categóricas...")

mappingLuminosity = {'DARK': 0, 'LOW_LIGHT': 1, 'LIGHT': 2}
mappingCloudiness = {'céu limpo': 0, 'céu claro': 0, 'céu pouco nublado': 1, 'algumas nuvens': 1, 'nuvens dispersas': 2, 'nuvens quebradas': 3, 'nuvens quebrados': 3, 'nublado': 4, 'tempo nublado': 4}
mappingRain = {'chuvisco fraco': 0, 'chuvisco e chuva fraca': 1, 'chuva fraca': 1, 'chuva leve': 1, 'aguaceiros fracos': 2, 'chuva': 2, 'aguaceiros': 3, 'chuva moderada': 3, 'chuva forte': 4, 'chuva de intensidade pesada': 5, 'chuva de intensidade pesado': 5, 'trovoada com chuva leve': 5, 'trovoada com chuva': 6}

for df in [dfTrain, dfTest]:
    df['LUMINOSITY'] = df['LUMINOSITY'].map(mappingLuminosity).fillna(-1)
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].map(mappingCloudiness).fillna(-1)
    df['AVERAGE_RAIN'] = df['AVERAGE_RAIN'].map(mappingRain).fillna(-1)

print("A criar features de ratio e interação...")

for df in [dfTrain, dfTest]:
    # Ratio de atraso
    df['TIME_DELAY_RATIO'] = df['AVERAGE_TIME_DIFF'] / df['AVERAGE_FREE_FLOW_TIME']
    df['TIME_DELAY_RATIO'] = df['TIME_DELAY_RATIO'].replace([np.inf, -np.inf], 0).fillna(0)

    # Ratio de fluxo livre
    df['FREE_FLOW_RATIO'] = df['AVERAGE_FREE_FLOW_TIME'] / (df['AVERAGE_FREE_FLOW_TIME'] + df['AVERAGE_TIME_DIFF'])
    df['FREE_FLOW_RATIO'] = df['FREE_FLOW_RATIO'].replace([np.inf, -np.inf], 0).fillna(0)

   
    df['is_raining'] = (df['AVERAGE_RAIN'] > -1).astype(int)  # 1 se chove, 0 se não
    df['is_dark'] = (df['LUMINOSITY'] == 0).astype(int)       # 1 se está escuro, 0 se não
    df['RUSH_HOUR_COM_CHUVA'] = df['isRushHour'] * df['is_raining']
    df['NOITE_COM_CHUVA'] = df['is_dark'] * df['is_raining']


print("A mapear a variável alvo...")

mappingSpeedDiff = {'Low': 0, 'Medium': 1, 'High': 2, 'Very_High': 3}
dfTrain['AVERAGE_SPEED_DIFF'] = dfTrain['AVERAGE_SPEED_DIFF'].map(mappingSpeedDiff)
dfTrain['AVERAGE_SPEED_DIFF'] = dfTrain['AVERAGE_SPEED_DIFF'].fillna(-1)



print("A finalizar a preparação dos dados...")

all_data = pd.concat([dfTrain.drop(columns=['AVERAGE_SPEED_DIFF']), dfTest], sort=False)

# One-Hot Encoding para 'Parte_do_Dia' e 'Estacao_do_Ano'
all_data_encoded = pd.get_dummies(all_data, columns=['Parte_do_Dia', 'Estacao_do_Ano'], drop_first=True)


dfTrain_final = all_data_encoded.iloc[:len(dfTrain)]
dfTest_final = all_data_encoded.iloc[len(dfTrain):]


dfTrain_final['AVERAGE_SPEED_DIFF'] = dfTrain['AVERAGE_SPEED_DIFF']


print("\nProcesso de feature engineering concluído!")
print("\nColunas do dfTrain após todas as transformações:")
print(dfTrain_final.columns.tolist())
print("\nAmostra do dfTrain final:")
print(dfTrain_final.head())

A criar features de Data e Hora...
A mapear features categóricas...
A criar features de ratio e interação...
A mapear a variável alvo...
A finalizar a preparação dos dados...

Processo de feature engineering concluído!

Colunas do dfTrain após todas as transformações:
['city_name', 'record_date', 'AVERAGE_FREE_FLOW_SPEED', 'AVERAGE_TIME_DIFF', 'AVERAGE_FREE_FLOW_TIME', 'LUMINOSITY', 'AVERAGE_TEMPERATURE', 'AVERAGE_ATMOSP_PRESSURE', 'AVERAGE_HUMIDITY', 'AVERAGE_WIND_SPEED', 'AVERAGE_CLOUDINESS', 'AVERAGE_PRECIPITATION', 'AVERAGE_RAIN', 'year', 'month', 'day', 'hour', 'dayOfWeek', 'isRushHour', 'IS_WEEKEND', 'is_holiday', 'hour_sin', 'hour_cos', 'dayOfWeek_sin', 'dayOfWeek_cos', 'TIME_DELAY_RATIO', 'FREE_FLOW_RATIO', 'is_raining', 'is_dark', 'RUSH_HOUR_COM_CHUVA', 'NOITE_COM_CHUVA', 'Parte_do_Dia_Manha', 'Parte_do_Dia_Tarde', 'Parte_do_Dia_Noite', 'Estacao_do_Ano_Outono', 'Estacao_do_Ano_Primavera', 'Estacao_do_Ano_Verao', 'AVERAGE_SPEED_DIFF']

Amostra do dfTrain final:
  city_name     

C:\Users\paulo\AppData\Local\Temp\ipykernel_4240\2407511615.py:105: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfTrain_final['AVERAGE_SPEED_DIFF'] = dfTrain['AVERAGE_SPEED_DIFF']


In [ ]:
import pandas as pd
import numpy as np
import holidays
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score 
! pip install holidays -q

print("A carregar os dados...")
dfTrain = pd.read_csv('training_data.csv', encoding='latin1')
dfTest = pd.read_csv('test_data.csv', encoding='latin1')


print("A executar a engenharia de features...")

for df in [dfTrain, dfTest]:
    df['record_date'] = pd.to_datetime(df['record_date'])
    df['hour'] = df['record_date'].dt.hour
    df['dayOfWeek'] = df['record_date'].dt.dayofweek
    df['month'] = df['record_date'].dt.month
    df['year'] = df['record_date'].dt.year
    df['isRushHour'] = ((df['hour'] >= 7) & (df['hour'] <= 10) | 
                        (df['hour'] >= 16) & (df['hour'] <= 21)).astype(int)
    df['IS_WEEKEND'] = (df['dayOfWeek'] >= 5).astype(int)
    anos = df['year'].unique()
    feriados_portugal = holidays.Portugal(years=anos)
    df['is_holiday'] = df['record_date'].dt.date.isin(feriados_portugal).astype(int)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
    df['dayOfWeek_sin'] = np.sin(2 * np.pi * df['dayOfWeek'] / 7.0)
    df['dayOfWeek_cos'] = np.cos(2 * np.pi * df['dayOfWeek'] / 7.0)
    df['TIME_DELAY_RATIO'] = df['AVERAGE_TIME_DIFF'] / df['AVERAGE_FREE_FLOW_TIME']
    df['FREE_FLOW_RATIO'] = df['AVERAGE_FREE_FLOW_TIME'] / (df['AVERAGE_FREE_FLOW_TIME'] + df['AVERAGE_TIME_DIFF'])
    df.replace([np.inf, -np.inf], 0, inplace=True)
    df.fillna(0, inplace=True)


print("A mapear features categóricas...")

mappingLuminosity = {'DARK': 0, 'LOW_LIGHT': 1, 'LIGHT': 2}
mappingCloudiness = {'céu limpo': 0, 'céu claro': 0, 'céu pouco nublado': 1, 'algumas nuvens': 1, 'nuvens dispersas': 2, 'nuvens quebradas': 3, 'nuvens quebrados': 3, 'nublado': 4, 'tempo nublado': 4}
mappingRain = {'chuvisco fraco': 0, 'chuvisco e chuva fraca': 1, 'chuva fraca': 1, 'chuva leve': 1, 'aguaceiros fracos': 2, 'chuva': 2, 'aguaceiros': 3, 'chuva moderada': 3, 'chuva forte': 4, 'chuva de intensidade pesada': 5, 'chuva de intensidade pesado': 5, 'trovoada com chuva leve': 5, 'trovoada com chuva': 6}

for df in [dfTrain, dfTest]:
    df['LUMINOSITY'] = df['LUMINOSITY'].map(mappingLuminosity).fillna(-1)
    df['AVERAGE_CLOUDINESS'] = df['AVERAGE_CLOUDINESS'].map(mappingCloudiness).fillna(-1)
    df['AVERAGE_RAIN'] = df['AVERAGE_RAIN'].map(mappingRain).fillna(-1)


print("A preparar os dataframes X e y finais...")

mappingSpeedDiff = {'Low': 0, 'Medium': 1, 'High': 2, 'Very_High': 3}
dfTrain['AVERAGE_SPEED_DIFF'] = dfTrain['AVERAGE_SPEED_DIFF'].map(mappingSpeedDiff).fillna(-1)


p1_inicio = pd.to_datetime('2018-09-12')
p1_fim = pd.to_datetime('2018-12-14')

p2_inicio = pd.to_datetime('2019-01-03')
p2_fim = pd.to_datetime('2019-04-05')

p3_inicio = pd.to_datetime('2019-04-23')
p3_fim = pd.to_datetime('2019-06-21')

# 3. Criar as condições booleanas para cada período
cond_p1 = (dfTrain['record_date'] >= p1_inicio) & (dfTrain['record_date'] <= p1_fim)
cond_p2 = (dfTrain['record_date'] >= p2_inicio) & (dfTrain['record_date'] <= p2_fim)
cond_p3 = (dfTrain['record_date'] >= p3_inicio) & (dfTrain['record_date'] <= p3_fim)

# 4. Combinar as condições
# Uma data é 'TEMPO_AULAS' se pertencer ao período 1 OU ao período 2 OU ao período 3
cond_total = cond_p1 | cond_p2 | cond_p3

dfTrain['TEMPO_AULAS'] = (cond_total).astype(int)

colunas_a_remover = [
    'AVERAGE_SPEED_DIFF',
    'AVERAGE_PRECIPITATION',
    'record_date', 
    'city_name',
    'hour',          
    'dayOfWeek'       
]
X = dfTrain.drop(columns=colunas_a_remover, errors='ignore')
y = dfTrain['AVERAGE_SPEED_DIFF']

X_teste = dfTest.drop(columns=[col for col in colunas_a_remover if col in dfTest.columns], errors='ignore')
X, X_teste = X.align(X_teste, join='inner', axis=1, fill_value=0)

print("\nA iniciar a otimização e treino do modelo...")

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

param_dist = {'max_features':np.arange(13,14),
              'n_estimators':[1000],
              'max_depth':[8],
              'min_samples_split':[5, 10],
              'min_samples_leaf':[2,4]
              }

# param_dist = {
#     'n_estimators': [100, 200,700,900,2000], 'max_depth': [5,10,15],
#     'min_samples_split': [5, 10, 20], 'min_samples_leaf': [2, 4],
#     'max_features': ['sqrt', 'log2']
# }

rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
random_search = GridSearchCV(rf, param_grid=param_dist, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=1)
random_search.fit(X_train, y_train)

best_params = random_search.best_params_
print("\nMelhores Hiperparâmetros Encontrados:", best_params)


print("\nA treinar o modelo final com todos os dados de treino...")
final_model = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1, **best_params)
final_model.fit(X, y)

# Fazer previsões no conjunto de treino completo para verificar a accuracy
y_pred_treino_final = final_model.predict(X)
# Calcular e imprimir a accuracy
print("\n--- PERFORMANCE NO CONJUNTO DE TREINO COMPLETO ---")
print(f"Accuracy no treino completo: {accuracy_score(y, y_pred_treino_final):.4f}")

print("\nA gerar o ficheiro de submissão...")
final_predictions_numeric = final_model.predict(X_teste)

mapping_reverse = {0: 'Low', 1: 'Medium', 2: 'High', 3: 'Very_High', -1: 'None'}
final_predictions_labels = [mapping_reverse[int(pred)] for pred in final_predictions_numeric]

row_ids = range(1, len(dfTest) + 1)
output = pd.DataFrame({'RowId': row_ids, 'Speed_Diff': final_predictions_labels})
output.to_csv('submission.csv', index=False)

print("\nFicheiro 'submission.csv' criado!")
print("\nDistribuição das classes na submissão:")
print(output['Speed_Diff'].value_counts())

A carregar os dados...
A executar a engenharia de features...
A mapear features categóricas...
A preparar os dataframes X e y finais...

A iniciar a otimização e treino do modelo...
Fitting 3 folds for each of 624 candidates, totalling 1872 fits

Melhores Hiperparâmetros Encontrados: {'max_depth': 8, 'max_features': np.int64(13), 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 1000}

A treinar o modelo final com todos os dados de treino...

--- PERFORMANCE NO CONJUNTO DE TREINO COMPLETO ---
Accuracy no treino completo: 0.8597

A gerar o ficheiro de submissão...

Ficheiro 'submission.csv' criado!

Distribuição das classes na submissão:
Speed_Diff
None         442
Low          366
Medium       307
High         276
Very_High    109
Name: count, dtype: int64


In [4]:
#printing accuracy on validation set

from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import KFold,cross_val_score
y_pred_val = final_model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred_val)
class_report_val = classification_report(y_val, y_pred_val)
print("\n--- PERFORMANCE NO CONJUNTO DE VALIDAÇÃO ---")
print("Relatório de Classificação no Conjunto de Validação:")
print(class_report_val)
print(f"\nAccuracy no conjunto de validação: {accuracy:.4f}")

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(final_model, X_train, y_train, cv=kf, scoring='accuracy', n_jobs=-1)

print(f"f1_weighted for each fold: {scores}")

print(f"f1_weighted: {scores.mean():.2f}")


--- PERFORMANCE NO CONJUNTO DE VALIDAÇÃO ---
Relatório de Classificação no Conjunto de Validação:
              precision    recall  f1-score   support

        -1.0       0.95      0.88      0.92       550
         0.0       0.72      0.87      0.79       355
         1.0       0.87      0.78      0.82       412
         2.0       0.84      0.85      0.85       266
         3.0       0.89      0.93      0.91       120

    accuracy                           0.85      1703
   macro avg       0.85      0.86      0.86      1703
weighted avg       0.86      0.85      0.86      1703


Accuracy no conjunto de validação: 0.8538
f1_weighted for each fold: [0.79843444 0.78571429 0.7739726  0.81409002 0.78354554]
f1_weighted: 0.79
